![image.png](https://i.imgur.com/4fN73lZ.png)

# Inverse RL: Recovering a Reward from Demonstrations (MaxEnt IRL)

Everything so far has been **forward RL**: given a reward, find a policy. **Inverse
RL** flips it — given expert *behavior*, recover a **reward** that explains it. This
is the strongest form of "the reward problem": we can't even write the reward down,
only demonstrate it.

IRL is **ill-posed** — many rewards explain the same behavior (e.g. $R\equiv 0$ makes
every policy optimal). **Maximum-Entropy IRL** (Ziebart et al., 2008) resolves the
ambiguity with a familiar principle: among all reward-induced trajectory
distributions consistent with the demonstrations, pick the **highest-entropy** one,
$$P_\theta(\tau) \propto \exp\!\Big(\textstyle\sum_t R_\theta(s_t)\Big).$$
This is the **same max-entropy principle** that justified SAC's $-\log\pi$ bonus on
Day 6 — here it does a different job (resolving reward ambiguity instead of driving
exploration).

The maximum-likelihood gradient turns out to be beautifully simple — **match feature
expectations**:
$$\nabla_\theta \mathcal{L} = \underbrace{\mathbb{E}_{\mathcal{D}}[\phi(s)]}_{\mu_{\text{expert}}} - \underbrace{\mathbb{E}_{p(\tau \mid R_\theta)}[\phi(s)]}_{\mu_{\text{expected}}}.$$
We implement this on a GridWorld: learn a reward from expert trajectories and check
it peaks where the true reward does.

> **Exercise version.** Fill the two `# TASK`s in `MaximumEntropy` — the pieces
> unique to MaxEnt IRL. A complete reference is in `Day-7_IRL_MaxEnt_Solution.ipynb`.
> - **TASK 1** — propagate the expected state-visitation frequencies.
> - **TASK 2** — the feature-count-matching gradient.

## Setup

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" numpy matplotlib tqdm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

## GridWorld

A $5\times5$ grid. **States** are numbered $s = x + y\cdot\text{grid\_size}$ with
$(0,0)$ at the bottom-left, so the last index $s = n_\text{states}-1$ is the
**top-right corner** — the single cell with reward $1$ (every other cell gives $0$).
Transitions are **windy**: the agent usually moves where it intends, but with
probability `wind` it slips to a random direction (walls keep it in place). The
"expert" `optimal_policy` simply walks toward the goal. IRL sees only these expert
trajectories — never the reward.

In [ ]:
class GridWorld:
    """A windy 5x5 grid. States are numbered s = x + y * grid_size, with (0,0) at
    the bottom-left, so s = n_states-1 is the top-right corner. The only rewarding
    cell is that top-right corner; the expert walks toward it."""
    def __init__(self, grid_size=5, wind=0.2):
        self.grid_size = grid_size
        self.wind = float(wind)                     # prob. of slipping to a random direction
        self.actions = [(1, 0), (0, 1), (-1, 0), (0, -1)]   # right, up, left, down (dx, dy)
        self.names = ["Right", "Up", "Left", "Down"]
        self.n_actions = len(self.actions)
        self.n_states = grid_size ** 2

        self.features = np.eye(self.n_states)       # one-hot state features
        self.dynamics = self.transition_probabilities()
        self.real_rewards = np.array([self.reward(s) for s in range(self.n_states)])
        self.state = 0

    def reward(self, state):
        """Reward is 1 only in the GOAL cell, 0 everywhere else. The goal is the last
        state index, n_states-1, which maps to the top-right corner (x = y = grid_size-1)."""
        goal_state = self.n_states - 1
        return 1 if state == goal_state else 0

    def transition_probabilities(self):
        """Build P[s_next, a, s] = probability of landing in s_next after taking
        action a in state s. With probability (1 - wind) the agent moves in the
        intended direction a; with probability `wind` it slips to a uniformly-random
        one of the 4 directions. A move that would leave the grid keeps it in place."""
        P = np.zeros((self.n_states, self.n_actions, self.n_states))
        for s in range(self.n_states):
            x, y = s % self.grid_size, s // self.grid_size
            for a in range(self.n_actions):
                for d in range(self.n_actions):          # d = the direction actually taken
                    prob = self.wind / self.n_actions + (1 - self.wind if d == a else 0.0)
                    dx, dy = self.actions[d]
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                        s_next = nx + ny * self.grid_size
                    else:
                        s_next = s                       # bumped into a wall -> stay put
                    P[s_next, a, s] += prob
        return P

    def reset(self):
        self.state = 0
        return self.state

    def step(self, a):
        probs = self.dynamics[:, a, self.state]
        self.state = np.random.choice(self.n_states, p=probs)
        return self.state

    def optimal_policy(self, state):
        """Hand-coded expert: move toward the top-right goal (increase whichever of
        x, y is currently behind)."""
        x, y = state % self.grid_size, state // self.grid_size
        if x > y:
            return 1          # move up  (0, 1)
        elif x < y:
            return 0          # move right (1, 0)
        else:
            return np.random.randint(2)   # on the diagonal: right or up

    def generate_trajectories(self, num, length, policy=None):
        if not policy:
            policy = self.optimal_policy
        trajs = []
        for _ in range(num):
            t = []
            state = self.reset()
            for _ in range(length):
                action = policy(state)
                state = self.step(action)
                t.append([state, action])
            trajs.append(t)
        return np.array(trajs)

## Soft value iteration

Given a candidate reward, we need the **expert model** it implies — the policy a
max-entropy agent would follow. That is a *softmax* over action values rather than a
hard $\arg\max$. Value iteration gives
$$Q(s,a) = \sum_{s'} P(s'\mid s,a)\,\big[R(s') + \gamma V(s')\big],\qquad
\pi(a\mid s) = \frac{e^{Q(s,a)}}{\sum_b e^{Q(s,b)}}.$$
The softmax (not argmax) is exactly what makes the model **maximum-entropy** — it
keeps every action possible in proportion to its value.

In [ ]:
def soft_value_iteration(env, rewards, gamma, tol=1e-4, max_iter=1000):
    """Return the max-entropy expert's policy for a given reward.

    Ordinary value iteration on the state values V, then a SOFTMAX over the action
    values Q(s, a) = sum_{s'} P(s'|s,a) [ reward(s') + gamma * V(s') ]. Using a
    softmax (instead of a hard argmax) is what makes this a *maximum-entropy* policy:
    every action stays possible, in proportion to its value.
    """
    n_states, n_actions = env.n_states, env.n_actions
    V = np.zeros(n_states)
    for _ in range(max_iter):
        Q = np.zeros((n_states, n_actions))
        for s in range(n_states):
            for a in range(n_actions):
                # env.dynamics[:, a, s] is P(s' | s, a) over all next states s'
                Q[s, a] = env.dynamics[:, a, s] @ (rewards + gamma * V)
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new

    # softmax over actions -> stochastic policy pi(a | s)
    Q = Q - Q.max(axis=1, keepdims=True)          # subtract max for numerical stability
    policy = np.exp(Q)
    policy /= policy.sum(axis=1, keepdims=True)
    return policy

## Maximum-Entropy IRL

Learn reward weights $\theta$ (so $R(s) = \theta^\top \phi(s)$) by **matching
feature expectations**. Two quantities:

- $\mu_\text{expert}$ — the average features the demonstrations actually visit.
- $\mu_\text{expected}$ — the features the *current* reward's policy would visit,
  obtained by rolling an **expected state-visitation** distribution forward through
  the policy and dynamics.

The max-entropy log-likelihood gradient is just their difference,
$$\nabla_\theta = \mu_\text{expert} - \mu_\text{expected},$$
which we ascend: raise the reward where the expert goes more than the model
expects, lower it where the model over-visits. At convergence the two match.

In [ ]:
class MaximumEntropy:
    """Maximum-Entropy IRL (Ziebart et al., 2008).

    Learns reward weights theta, giving R(s) = theta . phi(s), by MATCHING FEATURE
    EXPECTATIONS. The max-entropy log-likelihood gradient is just the difference
    between the expert's feature counts and those expected under the current reward:

        grad = mu_expert - mu_expected

    Ascend it until the two match.
    """
    def __init__(self, env, trajectories, features, lr=0.01, gamma=0.9):
        self.env = env
        self.trajectories = trajectories          # shape (num, length, 2): rows are [state, action]
        self.features = features                  # phi(s): one feature row per state
        self.lr = lr
        self.gamma = gamma
        self.theta = np.random.uniform(size=features.shape[1])

    def expert_feature_counts(self):
        """mu_expert: average features visited across the demonstration trajectories."""
        mu = np.zeros(self.features.shape[1])
        for traj in self.trajectories:
            for state, _action in traj:
                mu += self.features[state]
        return mu / len(self.trajectories)

    def expected_state_visitation(self, policy):
        """Expected number of visits to each state under `policy`.

        Start from the demonstrations' initial-state distribution, then roll the
        visitation distribution forward for `traj_len` steps through the policy and
        the transition dynamics, summing the per-step visitation.
        """
        num_traj, traj_len = self.trajectories.shape[0], self.trajectories.shape[1]

        mu0 = np.zeros(self.env.n_states)                 # initial-state distribution
        for traj in self.trajectories:
            mu0[traj[0, 0]] += 1.0
        mu0 /= num_traj

        mu = np.zeros((traj_len, self.env.n_states))
        mu[0] = mu0
        for t in range(1, traj_len):
            for s in range(self.env.n_states):
                for a in range(self.env.n_actions):
                    for s_prev in range(self.env.n_states):
                        # TASK 1: propagate expected state-visitation one step forward.
                        #   Mass at s_prev flows to s if the policy picks a there and the
                        #   dynamics move s_prev -> s. Accumulate over all (a, s_prev).
                        # HINT: mu[t, s] += mu[t-1, s_prev] * policy[s_prev, a] * dynamics[s, a, s_prev]
                        raise NotImplementedError("TASK 1: state-visitation propagation")
        return mu.sum(axis=0)                             # total expected visits per state

    def train(self, n_epochs, plot=False):
        mu_expert = self.expert_feature_counts()          # constant target
        for i in tqdm(range(n_epochs)):
            rewards = self.features @ self.theta
            policy = soft_value_iteration(self.env, rewards, self.gamma)
            mu_expected = self.expected_state_visitation(policy) @ self.features
            # TASK 2: the MaxEnt IRL gradient = match feature expectations.
            #   Raise reward where the EXPERT's counts exceed the model's expected counts.
            # HINT: grad = mu_expert - mu_expected
            grad = None
            if grad is None:
                raise NotImplementedError("TASK 2: feature-count-matching gradient")
            self.theta += self.lr * grad
            if plot and i % 50 == 0:
                plt.pcolor(rewards.reshape(self.env.grid_size, self.env.grid_size))
                plt.colorbar(); plt.title(f"recovered reward @ epoch {i}"); plt.show()
        return self.features @ self.theta

## Learn a reward from demonstrations

In [ ]:
grid_size = 5
wind_rate = 0.3
num_trajectories = 50
len_trajectory = 30
n_epochs = 500
lr = 0.01
gamma = 0.9

np.random.seed(0)
gw = GridWorld(grid_size, wind_rate)
trajectories = gw.generate_trajectories(num_trajectories, len_trajectory)
features = np.eye(gw.n_states)                    # one-hot state features

me = MaximumEntropy(gw, trajectories, features, lr, gamma)
recovered = me.train(n_epochs, plot=False)

## True vs recovered reward

The recovered reward should peak at the same cell as the true reward (top-right) and
fall off smoothly toward it — even though IRL only ever saw trajectories.

In [ ]:
true_rewards = np.array([gw.reward(s) for s in range(gw.n_states)]).reshape(grid_size, grid_size)
rec = recovered.reshape(grid_size, grid_size)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im0 = ax[0].pcolor(true_rewards); ax[0].set_title("true reward"); fig.colorbar(im0, ax=ax[0])
im1 = ax[1].pcolor(rec);          ax[1].set_title("recovered reward (MaxEnt IRL)"); fig.colorbar(im1, ax=ax[1])
plt.tight_layout(); plt.show()

print("true reward argmax state:     ", int(true_rewards.argmax()))
print("recovered reward argmax state:", int(rec.argmax()))

## Where this goes next

The same `MaximumEntropy` recovers rewards on much richer problems — all that
changes is the **feature map** $\phi(s)$ (here one-hot per state; in general,
hand-designed or learned features). MaxEnt IRL scales to continuous domains and
deep feature networks (Ziebart 2008; Wulfmeier 2015). The next rung of the reward
problem is learning rewards from **preferences** instead of demonstrations
(Bradley–Terry) — the direct ancestor of **RLHF (Day 9)**.

## Takeaways

- **IRL learns a reward from demonstrations** — for tasks where behavior is easy to
  show but the reward is hard to write (driving, manipulation).
- **MaxEnt IRL** resolves IRL's ill-posedness with the *same max-entropy principle*
  as SAC (Day 6): match the expert's feature expectations, stay maximally
  non-committal otherwise.
- The next rung is learning rewards from **preferences** instead of demonstrations
  (Bradley–Terry) — the direct ancestor of **RLHF (Day 9)**.